# Multi-Model Content Pipeline — Ray Serve on Anyscale

This notebook demonstrates deploying **4 independently-scaling ML deployments** as a single Ray Serve application on Anyscale.

| Deployment | Resource | Model | Latency |
|---|---|---|---|
| **ContentFilter** | CPU only | Regex/rules | ~1 ms |
| **SentimentClassifier** | 0.25 GPU | DistilBERT (67M) | ~10 ms |
| **Summarizer** | 0.25 GPU | DistilBART-CNN (306M) | ~200 ms |
| **ContentPipeline** | CPU only | Orchestrator | N/A |

**Key idea**: Each deployment scales to its own bottleneck. CPU traffic spikes don't waste GPU resources.

---
## 1. Architecture

```
Client
  │
  ▼
ContentPipeline (CPU · ingress · 1-5 replicas)
  │
  ├──▶ ContentFilter (CPU · 1-8 replicas)
  │       └── Reject? → Return early (no GPU cost)
  │
  └──▶ asyncio.gather (parallel fan-out)
        ├──▶ SentimentClassifier (0.25 GPU · 1-4 replicas)
        └──▶ Summarizer          (0.25 GPU · 1-4 replicas)
```

**Infrastructure** (Anyscale auto-provisioned):
- Head node: `16CPU-64GB` (control plane, `Standard_D16s_v5`)
- GPU workers: `A10-24G` (both GPU models share via fractional GPU)
- CPU workers: `16CPU-64GB` (filter + orchestrator, `Standard_D16s_v5`)

---
## 2. Setup & Dependencies

In [1]:
import os

PROJECT_DIR = "/home/ray/default/Multi-Modal-Template"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /home/ray/default/Multi-Modal-Template


In [2]:
!pip install -q requests pyyaml

Successfully registered `requests, pyyaml` packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_d8z3s9fkwjaa1pt8agy86v9hwk/prj_3gtt4pb1x12dii9llldr5dmctd/workspaces/expwrk_z1upnwssjjs4j9rk5eivc8jwt2?workspace-tab=dependencies


In [3]:
import json
import time
import concurrent.futures
import subprocess

import requests
import yaml

---
## 3. Inspect the Service Configuration

In [4]:
with open("service.yaml") as f:
    config = yaml.safe_load(f)

print("Service name:", config["name"])
print("Image:", config["image_uri"])
print("\nDeployments:")
for d in config["applications"][0]["deployments"]:
    gpu = d.get("ray_actor_options", {}).get("num_gpus", 0)
    asc = d.get("autoscaling_config", {})
    print(f"  {d['name']:25s}  GPU: {gpu}  replicas: {asc.get('min_replicas',1)}-{asc.get('max_replicas','?')}")

print("\nCompute config:")
cc = config["compute_config"]
if isinstance(cc, str):
    print(f"  Named reference: {cc}")
    # Look for the sibling YAML that defines this named compute config
    candidate = "compute_config_azure.yaml"
    if os.path.exists(candidate):
        with open(candidate) as f:
            inline = yaml.safe_load(f)
        head = inline.get("head_node", {}).get("instance_type", "?")
        n_workers = len(inline.get("worker_nodes", []) or [])
        print(f"  Source file:     {candidate}")
        print(f"  Head node:       {head}")
        print(f"  Worker groups:   {n_workers}")
else:
    print(f"  Head node: {cc['head_node']['instance_type']}")
    print(f"  Workers: auto_select = {cc.get('auto_select_worker_config', False)}")

Service name: multi-model-content-pipeline-2298227
Image: anyscale/ray-llm:2.56.0-py312-cu130

Deployments:
  ContentPipeline            GPU: 0  replicas: 1-5
  ContentFilter              GPU: 0  replicas: 1-8
  SentimentClassifier        GPU: 0.25  replicas: 1-4
  Summarizer                 GPU: 0.25  replicas: 1-4

Compute config:
  Named reference: multi-modal-2298227:2
  Source file:     compute_config_azure.yaml
  Head node:       8CPU-32GB
  Worker groups:   2


---
## 4. Inspect the Application Code

The 4 deployments are defined in `serve_app.py`. Here's a quick look at each:

In [5]:
with open("serve_app.py") as f:
    source = f.read()

import re

classes = re.findall(r"class (\w+):", source)
print("Deployments defined in serve_app.py:")
for cls in classes:
    print(f"  - {cls}")

print(f"\nTotal lines: {len(source.splitlines())}")
print("\nEntry point (last 6 lines):")
for line in source.splitlines()[-6:]:
    print(f"  {line}")

Deployments defined in serve_app.py:
  - ContentFilter
  - SentimentClassifier
  - Summarizer
  - ContentPipeline

Total lines: 404

Entry point (last 6 lines):
  
  app = ContentPipeline.bind(
      content_filter=ContentFilter.bind(),
      sentiment_classifier=SentimentClassifier.bind(),
      llm_summarizer=Summarizer.bind(),
  )


### 4a. ContentFilter (CPU)
- Regex-based PII detection & redaction
- Blocklist check
- ~1 ms per request, scales to 8 replicas
- **Why separate?** Acts as a cheap gate — rejected content never reaches GPU

### 4b. SentimentClassifier (0.25 GPU)
- DistilBERT (`distilbert-base-uncased-finetuned-sst-2-english`), 67M params
- HuggingFace `pipeline("sentiment-analysis")` on CUDA
- 0.25 GPU = 4 replicas can share 1 physical A10

### 4c. Summarizer (0.25 GPU)
- DistilBART-CNN (`sshleifer/distilbart-cnn-12-6`), 306M params
- HuggingFace `pipeline("summarization")` on CUDA
- Also 0.25 GPU — both GPU models share the same A10

### 4d. ContentPipeline (CPU Orchestrator)
- FastAPI ingress with `/analyze` and `/health` endpoints
- Calls filter first, then fans out to both GPU models via `asyncio.gather()`

---
## 5. Deploy to Anyscale

This cell deploys the service. Anyscale will:
1. Build a container image with our dependencies
2. Provision head + worker nodes (auto-selected)
3. Start all 4 Ray Serve deployments
4. Expose an HTTPS endpoint with auth token

In [6]:
result = subprocess.run(
    ["anyscale", "service", "deploy", "-f", "service.yaml"],
    capture_output=True, text=True, cwd=PROJECT_DIR
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

### 5a. Extract service URL and token from deploy output

In [7]:
SERVICE_NAME = config["name"]

status_result = subprocess.run(
    ["anyscale", "service", "status", "--name", SERVICE_NAME, "--json"],
    capture_output=True, text=True
)
print(status_result.stdout)

# Parse structured JSON output (robust against terminal line wrapping).
try:
    status_json = json.loads(status_result.stdout)
except json.JSONDecodeError:
    status_json = {}
    print("Could not parse status JSON. STDERR:", status_result.stderr)

SERVICE_URL = status_json.get("query_url")
AUTH_TOKEN = status_json.get("query_auth_token")

print(f"\nService URL: {SERVICE_URL}")
if AUTH_TOKEN:
    print(f"Auth Token:  {AUTH_TOKEN[:10]}...")
else:
    print("Auth Token: not found")

{
  "name": "multi-model-content-pipeline-2298227",
  "id": "service2_7rxd7239gj8jd2p85hxfbtdcne",
  "state": "STARTING",
  "query_url": "https://multi-model-content-pipeline-2298227-dmctd.cld-d8z3s9fkwjaa1pt8.s.anyscaleuserdata.com",
  "creator": "naila@anyscale.com",
  "query_auth_token": "-Y1GKjMpPxjZWDMREseq_Y_3kWjYIjvKp1SnA7NKg0k",
  "primary_version": {
    "id": "1294b8bj",
    "name": "v-pb5cc0dcw9",
    "state": "STARTING",
    "weight": 100,
    "created_at": "2026-06-30T07:36:44.091726+00:00"
  }
}


Service URL: https://multi-model-content-pipeline-2298227-dmctd.cld-d8z3s9fkwjaa1pt8.s.anyscaleuserdata.com
Auth Token:  -Y1GKjMpPx...


### 5b. Wait for service to be RUNNING

In [8]:
print(f"Waiting for '{SERVICE_NAME}' to become RUNNING (up to 10 min)...")
print("This includes: container build, node provisioning, model downloads, warmup.\n")

timeout = 600
start = time.time()
last_state = ""
terminal_states = {"RUNNING", "SYSTEM_FAILURE", "TERMINATED", "UNHEALTHY"}

while time.time() - start < timeout:
    r = subprocess.run(
        ["anyscale", "service", "status", "--name", SERVICE_NAME, "--json"],
        capture_output=True, text=True
    )
    try:
        state = json.loads(r.stdout).get("state", "UNKNOWN")
    except json.JSONDecodeError:
        state = "UNKNOWN"

    elapsed = int(time.time() - start)
    if state != last_state:
        print(f"  [{elapsed:3d}s] State: {state}")
        last_state = state

    if state == "RUNNING":
        print(f"\nService is RUNNING after {elapsed}s!")
        break
    if state in terminal_states:
        print(f"\nService entered {state}. Check Anyscale console for details.")
        break

    time.sleep(15)
else:
    print(f"\nTimed out after {timeout}s. Check: anyscale service status --name {SERVICE_NAME}")

Waiting for 'multi-model-content-pipeline-2298227' to become RUNNING (up to 10 min)...
This includes: container build, node provisioning, model downloads, warmup.

  [  3s] State: STARTING


---
## 6. Test the Live Service

Helper function to call the deployed endpoint:

In [ ]:
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AUTH_TOKEN}",
}

def call_endpoint(path, method="GET", payload=None):
    """Call the Anyscale service endpoint and return the JSON response."""
    url = f"{SERVICE_URL}{path}"
    if method == "GET":
        resp = requests.get(url, headers=HEADERS, timeout=30)
    else:
        resp = requests.post(url, headers=HEADERS, json=payload, timeout=30)
    return resp.status_code, resp.json()

print("Helper ready. Endpoint:", SERVICE_URL)

: 

### 6a. Health Check

In [ ]:
status, body = call_endpoint("/health")
print(f"Status: {status}")
print(json.dumps(body, indent=2))

### 6b. Normal Text (passes all stages)

In [ ]:
text = (
    "Ray Serve makes it easy to deploy multiple ML models as a single "
    "application. Each deployment scales independently based on its own "
    "load profile — CPU-bound business logic, lightweight GPU models, and "
    "heavy LLM inference all coexist without interfering with each other. "
    "This composability is a key advantage over monolithic serving frameworks."
)

start = time.time()
status, body = call_endpoint("/analyze", method="POST", payload={"text": text})
latency = time.time() - start

print(f"Status: {status}  |  Latency: {latency:.2f}s")
print(json.dumps(body, indent=2))

### 6c. Text with PII (should redact and proceed)

In [ ]:
text_pii = (
    "Please contact John at john.doe@example.com or call 555-123-4567 "
    "for more information about the quarterly earnings report. The company "
    "exceeded revenue expectations by 15% this quarter."
)

start = time.time()
status, body = call_endpoint("/analyze", method="POST", payload={"text": text_pii})
latency = time.time() - start

print(f"Status: {status}  |  Latency: {latency:.2f}s")
print(json.dumps(body, indent=2))

if body.get("filter", {}).get("pii_detected"):
    print("\n✓ PII was detected and redacted before reaching GPU models")

: 

### 6d. Blocked Content (rejected — no GPU cost)

In [ ]:
start = time.time()
status, body = call_endpoint(
    "/analyze", method="POST",
    payload={"text": "This is a spam message trying to scam people."}
)
latency = time.time() - start

print(f"Status: {status}  |  Latency: {latency:.3f}s")
print(json.dumps(body, indent=2))
print("\n✓ Rejected by ContentFilter — GPU deployments were never called")

### 6e. Too-Short Text (rejected)

In [ ]:
status, body = call_endpoint("/analyze", method="POST", payload={"text": "Hi"})
print(f"Status: {status}")
print(json.dumps(body, indent=2))

---
## 7. Throughput Test — Observe Independent Scaling

Send concurrent requests to trigger autoscaling. Watch the Ray Dashboard (linked in Anyscale console) to see replicas scale independently.

In [ ]:
NUM_REQUESTS = 20

payload = {
    "text": (
        "Artificial intelligence is transforming industries from healthcare to "
        "finance. Machine learning models can now process vast amounts of data "
        "to make predictions, automate decisions, and generate content. "
        "The rapid advancement of large language models has opened new possibilities "
        "for natural language understanding and generation at scale."
    )
}

def send_one(i):
    t0 = time.time()
    resp = requests.post(
        f"{SERVICE_URL}/analyze", headers=HEADERS, json=payload, timeout=60
    )
    return i, resp.status_code, time.time() - t0

print(f"Sending {NUM_REQUESTS} concurrent requests...\n")
overall_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_REQUESTS) as pool:
    futures = [pool.submit(send_one, i) for i in range(NUM_REQUESTS)]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

overall = time.time() - overall_start

for i, code, latency in sorted(results):
    bar = "█" * int(latency * 5)
    print(f"  Request {i:2d}:  {code}  {latency:5.2f}s  {bar}")

latencies = [r[2] for r in results]
print(f"\nWall-clock time: {overall:.2f}s")
print(f"Throughput:      {NUM_REQUESTS / overall:.1f} req/s")
print(f"Avg latency:     {sum(latencies)/len(latencies):.2f}s")
print(f"P50 latency:     {sorted(latencies)[len(latencies)//2]:.2f}s")
print(f"P99 latency:     {sorted(latencies)[int(len(latencies)*0.99)]:.2f}s")

: 

---
## 8. Check Service Logs

Pull the controller logs to see model loading and replica activity:

In [ ]:
result = subprocess.run(
    ["anyscale", "service", "controller_logs", "--name", SERVICE_NAME],
    capture_output=True, text=True
)
logs = result.stdout

for line in logs.splitlines()[-50:]:
    print(line)

---
## 9. Service Status Summary

In [ ]:
result = subprocess.run(
    ["anyscale", "service", "status", "--name", SERVICE_NAME],
    capture_output=True, text=True
)
print(result.stdout)

---
## 10. Cleanup (Optional)

Terminate the service when you're done to stop incurring costs:

In [ ]:
# Uncomment to terminate:
# !anyscale service terminate --name {SERVICE_NAME}
print(f"To terminate: anyscale service terminate --name {SERVICE_NAME}")

---
## Manual Steps Reference

If you prefer to run this from the terminal instead of the notebook:

```bash
# 1. Navigate to the project
cd /home/ray/default/multi_model_pipeline_ray_serve_20260210_155304

# 2. Deploy the service
anyscale service deploy -f service.yaml

# 3. Wait for RUNNING state
anyscale service wait --name multi-model-content-pipeline --timeout-s 600

# 4. Check status
anyscale service status --name multi-model-content-pipeline

# 5. Test with curl
curl -H "Authorization: Bearer YOUR_TOKEN" \
     -H "Content-Type: application/json" \
     -d '{"text": "Ray Serve enables independent scaling of ML models."}' \
     https://YOUR_SERVICE_URL/analyze

# 6. Or use the test client
python client.py --url https://YOUR_SERVICE_URL --token YOUR_TOKEN

# 7. Throughput test
python client.py --url https://YOUR_SERVICE_URL --token YOUR_TOKEN --throughput 20

# 8. View logs
anyscale service controller_logs --name multi-model-content-pipeline

# 9. Terminate
anyscale service terminate --name multi-model-content-pipeline
```